# Steps 3–5 — CrewAI Agents

🇬🇧 **English** (this notebook)

Single Agent → Multi-Agent → RAG + Tools, in one file. These three steps all edit the *same* `agents.yaml`/`tasks.yaml`/`crew.py` progressively — step 4 adds to what step 3 built, step 5 adds to step 4 — so they're combined here rather than split into three separate files. Each section below has its own "run" cell so you can see the crew's output at each stage as you build it up.

Run the **Setup** cell once, then work through Step 3, Step 4, and Step 5 in order, editing the real `agents.yaml`/`tasks.yaml`/`crew.py` files between each run (a notebook cell can't do that part).

## Setup

Run this once before any of the three steps below.

In [ ]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from research_crew.crew import ResearchCrew

load_dotenv()
os.makedirs("output", exist_ok=True)

# Same topic as steps 1-2. Keep it the same across all 5 steps.
TOPIC = "Explain the EU AI Act in simple terms for a 10-year-old in a short paragraph."

---

# Step 3 — Single Agent

Move from a hand-written prompt to a CrewAI `Agent` and `Task`. The `role`, `goal`, and `backstory` you write in `agents.yaml` are still just a system prompt under the hood — CrewAI assembles it for you. What the framework adds is the loop: the agent reasons in steps before producing output, can call tools (step 5), and retries on failure. One agent, one task.

## Background

The core loop that makes an agent an agent — alternating between reasoning about what to do next and taking an action (calling a tool, reading a result, updating a plan) — was introduced in:

> Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models*. ICLR 2023. [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)

ReAct (Reason + Act) is the pattern CrewAI agents follow: the model thinks ("I need to find X"), acts (calls a tool), observes the result, thinks again, and repeats until it can produce a final answer. This is what separates an agent from a single prompt call — the loop.

The broader observation that LLM-based agents benefit from an explicit module structure — profile (who the agent is), memory, planning, action — was systematized in:

> Wang, L., Ma, C., Feng, X., Zhang, Z., Yang, H., Zhang, J., Chen, Z., Tang, J., Chen, X., Lin, Y., Zhao, W. X., Wei, Z., & Wen, J. (2023). *A Survey on Large Language Model based Autonomous Agents*. [arXiv:2308.11432](https://arxiv.org/abs/2308.11432)

![Unified framework for LLM-based autonomous agents: Profile, Memory, Planning, Action modules](../assets/agentsurvey-wang2023-fig2.png)
*Figure 2 from Wang et al. (2023). Reproduced for educational use in this course.*

In CrewAI terms: `role`/`goal`/`backstory` in `agents.yaml` = **Profile**; `tools` + the ReAct task loop = **Action**.

## In this repo

| File | What to change |
| --- | --- |
| [src/research_crew/config/agents.yaml](../../src/research_crew/config/agents.yaml) | Define ONE agent for your topic — role, goal, backstory |
| [src/research_crew/config/tasks.yaml](../../src/research_crew/config/tasks.yaml) | Define ONE task — description, expected_output, agent |
| [src/research_crew/crew.py](../../src/research_crew/crew.py) | Keep only one `@agent` and one `@task` method |

The template already has two agents (`researcher` and `analyst`). For this step, reduce it to one — comment out or remove the analyst agent and its task.

## Your task

1. Open `agents.yaml`. Replace the existing agent with your own: give it a `role`, `goal`, and `backstory` suited to your topic.

2. Open `tasks.yaml`. Replace the existing task with one that fits your agent and topic: write a `description` and an `expected_output`.

3. In `crew.py`, keep only one `@agent` and one `@task` method (remove or comment out the analyst).

4. Run the cell below (or, from the terminal, `uv run research_crew`).

In [ ]:
result = ResearchCrew().crew().kickoff(inputs={"topic": TOPIC})
display(Markdown(result.raw))

5. Read the verbose log above — this is the first time you can see the agent's internal reasoning, not just the final answer. Does the agent break the task into sub-steps? Does the output feel different from what step 2 produced?

6. Fill in the **Step 3** section of `EVALUATION.md`.

## Stretch goal

Look at the verbose log's "Final Answer" alongside the agent's intermediate reasoning. Find one place where the reasoning and the conclusion seem inconsistent — where the agent reasons toward one thing and writes something slightly different. What does this tell you about trusting chain-of-thought?

**→ Interim submission is due after this step.** Submit the state of `main` after merging `sprint-3`. See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for exactly what's expected.

---

# Step 4 — Multi-Agent

Add a second agent with a different, complementary role. The first agent's output is passed to the second via `context:` in `tasks.yaml`. This is multi-agent in the simplest sense: role specialization with task chaining, not dynamic delegation.

## Background

A single agent can do multiple things, but it can't hold two genuinely different epistemic roles at the same time — it can't be simultaneously credulous (collecting everything) and skeptical (questioning what it found). Two agents let you encode that tension into the architecture. The seminal demonstration of agents collaborating through conversation was:

> Wu, Q., Bansal, G., Zhang, J., Wu, Y., Li, B., Zhu, E., Jiang, L., Zhang, X., Zhang, S., Liu, J., Awadallah, A. H., White, R. W., Burger, D., & Wang, C. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)

![AutoGen: conversable agents solving tasks in joint or hierarchical patterns](../assets/autogen-wu2023-fig1.png)
*Figure 1 from Wu et al. (2023). Reproduced for educational use in this course.*

## In this repo

| File | What to change |
| --- | --- |
| [src/research_crew/config/agents.yaml](../../src/research_crew/config/agents.yaml) | Add a second agent with a role that complements the first |
| [src/research_crew/config/tasks.yaml](../../src/research_crew/config/tasks.yaml) | Add a second task; link it to the first with `context:` |
| [src/research_crew/crew.py](../../src/research_crew/crew.py) | Add a second `@agent` and `@task` method |

The `context:` field in `tasks.yaml` is how the second agent receives the first agent's output:

```yaml
second_task:
  description: ...
  expected_output: ...
  agent: agent_2
  context:
    - first_task
```

## Your task

1. In `agents.yaml`, add a second agent with a role that's genuinely different from the first — not the same job with a different label.

2. In `tasks.yaml`, add a second task assigned to that agent. Add `context: - first_task` so it receives the first agent's output.

3. In `crew.py`, add a second `@agent` method and a second `@task` method.

4. Run the cell below (or, from the terminal, `uv run research_crew`).

In [ ]:
result = ResearchCrew().crew().kickoff(inputs={"topic": TOPIC})
display(Markdown(result.raw))

5. Read both agents' verbose output above. Does the second agent actually build on or push back against the first, or does it mostly repackage the same content? Does the final output feel qualitatively different from step 3?

6. **Experiment**: remove the `context:` line from the second task in `tasks.yaml` and run this cell again. What happens to the second agent's output when it can no longer see the first agent's work?

7. Fill in the **Step 4** section of `EVALUATION.md`.

## Stretch goal

Add a third agent whose absence you'd actually notice — a critic, a translator for a non-expert audience, or a validator. Run it and check whether the output meaningfully changes. If it doesn't, ask yourself why.

---

# Step 5 — RAG + Tools

Add two forms of external grounding to your step 4 crew. **Tools** let an agent retrieve live information at runtime; the agent decides when and with what query to call the tool. **RAG** lets the crew retrieve from a document you provide; chunks are embedded and the most relevant ones are injected into context automatically. Both address the same root limitation in steps 1–4: the LLM's knowledge is frozen at training time.

## Background

### Tools

> Schick, T., Dwivedi-Yu, J., Dessì, R., Raileanu, R., Lomeli, M., Zettlemoyer, L., Cancedda, N., & Scialom, T. (2023). *Toolformer: Language Models Can Teach Themselves to Use Tools*. [arXiv:2302.04761](https://arxiv.org/abs/2302.04761)

![Toolformer inserting API calls into its own generated text](../assets/toolformer-schick2023-fig1.png)
*Figure 1 from Schick et al. (2023). Reproduced for educational use in this course.*

### RAG

> Lewis, P., Perez, E., Piktus, A., Petroni, F., Karpukhin, V., Goyal, N., Küttler, H., Lewis, M., Yih, W., Rocktäschel, T., Riedel, S., & Kiela, D. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)

![RAG: query encoder and retriever feed top-k documents into a generator](../assets/rag-lewis2020-fig1.png)
*Figure 1 from Lewis et al. (2020). Reproduced for educational use in this course.*

**One practical detail**: embeddings use a separate model from the chat LLM. This crew uses Groq for chat and Gemini for embeddings — two different services, two different API keys, two different rate limits. The `embedder` block in `crew.py` is already configured; you just need `GEMINI_API_KEY` set.

## In this repo

| File | What to change |
| --- | --- |
| [src/research_crew/crew.py](../../src/research_crew/crew.py) | Add `tools=[SerperDevTool()]` to the relevant agent; add `knowledge_sources=[...]` to the `Crew` |
| [src/research_crew/knowledge_source_example.py](../../src/research_crew/knowledge_source_example.py) | Template for `build_knowledge_sources()` — import and call it in `crew.py` |
| `knowledge/` | Add your own document here (`.txt` or `.pdf`) |

The `embedder` block is already in `crew.py` — it's what makes RAG work without an OpenAI key.

## Your task

1. Add a knowledge document to `knowledge/` that's relevant to your topic — a `.txt` file with background information works fine to start.

2. In `crew.py`, import `build_knowledge_sources` from `knowledge_source_example.py` (or write the source inline), point it at your file, and pass it to the `Crew` constructor:
   ```python
   knowledge_sources=build_knowledge_sources()
   ```

3. Add `SerperDevTool()` to the agent that searches for information, in `crew.py`:
   ```python
   tools=[SerperDevTool()]
   ```

4. Update that agent's task description in `tasks.yaml` to include a question only answerable from your knowledge document. Run the cell below (or, from the terminal, `uv run research_crew`).

In [ ]:
result = ResearchCrew().crew().kickoff(inputs={"topic": TOPIC})
display(Markdown(result.raw))

5. Now run again **without** the knowledge source (comment it out). Does the agent still answer the document-specific question correctly, or does it admit it doesn't know? That comparison is the point of RAG.

6. Check the verbose log above: can you see when the agent calls the search tool, and what query it used?

7. Fill in the **Step 5** section of `EVALUATION.md` and your final recommendation.

## Stretch goal

Add a PDF instead of a plain text file:
```python
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource
PDFKnowledgeSource(file_paths=["your_document.pdf"])
```
Ask a question whose answer appears on a specific page. Does the agent retrieve it?

---

**This is your final submission.** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full grading rubric and exactly what to submit.